# L3 — Chatbot UI

## What You'll Learn

- How to authenticate users via Amazon Cognito and extract JWT claims
- How to invoke the AgentCore Harness with a user's actor ID for per-user memory scoping
- How to stream the Harness response token-by-token for a responsive UI
- How to launch the Streamlit chatbot that ties authentication and orchestration together

## Overview

This notebook wires the full AnyCompany customer support system into a Streamlit chatbot. Users log in with their Cognito credentials, and the chatbot passes their `customer_id` as the `actorId` to the Harness — scoping long-term memory (preferences, past session summaries) per user. The Harness then discovers and delegates to the Order and Refund agents as needed.

## Architecture

```
  User (browser)
       │  username + password
       ▼
  Cognito User Pool ──► JWT IdToken (contains customer_id, tier)
       │
       ▼
  Streamlit Chatbot (chatbot_app.py)
       │  actorId = customer_id from JWT
       ▼
  AgentCore Harness (invoke_harness)
       │
       ├──► Order Agent  (via Registry + Gateway)
       └──► Refund Agent (via Registry + Gateway)
```

## Steps

1. Load configuration from SSM
2. Verify Cognito configuration
3. Define the authentication helper
4. Define the Harness invocation helper
5. Launch the Streamlit chatbot UI

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- `1_pre-requisites.ipynb` completed — Cognito User Pool and test users must exist
- `4_orchestrator_agent.ipynb` completed — `harness_arn` must be in SSM
- `boto3`, `gradio`, `streamlit`, and `pyjwt[crypto]` (installed in the first code cell)

Install required packages.

In [ ]:
%pip install --quiet boto3==1.43.0 gradio==4.44.0 requests==2.32.4 "pyjwt[crypto]==2.13.0"

## Step 1: Load Configuration from SSM

Fetch the Harness ARN, Gateway URL, and Cognito IDs from SSM.

In [ ]:
import boto3
import json
import uuid
import time
import os

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region
SSM_PREFIX = "/anycompany/agentcore"

ssm = boto3.client("ssm", region_name=REGION)

def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        if default is not None:
            return default
        raise

REGISTRY_ID = get_ssm(f"{SSM_PREFIX}/registry_id")
HARNESS_ARN = get_ssm(f"{SSM_PREFIX}/harness_arn", default="")
GATEWAY_URL = get_ssm(f"{SSM_PREFIX}/gateway_url", default="")

# If harness ARN not in SSM, look it up by name
if not HARNESS_ARN:
    agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
    harnesses = agentcore_control.list_harnesses()
    for h in harnesses.get("harnesses", []):
        if h.get("harnessName") == "anycompany_orchestrator_v3":
            HARNESS_ARN = h["arn"]
            break
    if not HARNESS_ARN:
        raise RuntimeError("Harness not found. Run Notebook 5 first to create the orchestrator.")

# Cognito config — set these after creating your user pool
COGNITO_USER_POOL_ID = get_ssm(f"{SSM_PREFIX}/cognito_user_pool_id", default="")
COGNITO_CLIENT_ID = get_ssm(f"{SSM_PREFIX}/cognito_client_id", default="")

print(f"Region:       {REGION}")
print(f"Registry ID:  {REGISTRY_ID}")
print(f"Harness ARN:  {HARNESS_ARN}")
print(f"Gateway URL:  {GATEWAY_URL}")
print(f"Cognito Pool: {COGNITO_USER_POOL_ID or '(not set)'}")
print(f"Cognito App:  {COGNITO_CLIENT_ID or '(not set)'}")

## Step 2: Verify Cognito Configuration

Confirm the Cognito User Pool and app client IDs are available before proceeding.

In [ ]:
cognito = boto3.client("cognito-idp", region_name=REGION)

if not COGNITO_USER_POOL_ID or not COGNITO_CLIENT_ID:
    raise RuntimeError("Cognito not configured. Run Notebook 1 first.")

print(f"Cognito User Pool: {COGNITO_USER_POOL_ID}")
print(f"Cognito Client:    {COGNITO_CLIENT_ID}")
print("Ready for authentication.")

## Step 3: Authentication Helper

Define `authenticate_user()` (Cognito USER_PASSWORD_AUTH flow) and `decode_token_claims()` (JWT decode for display). Test with `gold_customer`.

In [ ]:
def authenticate_user(username: str, password: str) -> dict:
    """Authenticate user with Cognito and return tokens."""
    try:
        response = cognito.initiate_auth(
            ClientId=COGNITO_CLIENT_ID,
# PRODUCTION REQUIREMENT: Replace USER_PASSWORD_AUTH with federated identity (SAML/OIDC).
# Shared passwords do not meet enterprise security requirements.
# See: https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools-identity-federation.html
            AuthFlow="USER_PASSWORD_AUTH",
            AuthParameters={
                "USERNAME": username,
                "PASSWORD": password,
            },
        )
        result = response["AuthenticationResult"]
        return {
            "success": True,
            "id_token": result["IdToken"],
            "access_token": result["AccessToken"],
            "refresh_token": result.get("RefreshToken", ""),
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


def decode_token_claims(id_token: str) -> dict:
    """Decode JWT claims without verification (for display purposes)."""
    import jwt
    return jwt.decode(id_token, options={"verify_signature": False})


# Retrieve the workshop test password from SSM (stored as SecureString).
# get_ssm() passes WithDecryption=True so we get the plaintext back.
USER_PASSWORD = get_ssm(f"{SSM_PREFIX}/user_password")

# Test authentication
# ⚠️ WORKSHOP ONLY — disposable Cognito test user; password fetched from SSM.
# In production, use federated identity (SAML/OIDC) instead of shared passwords.
test_auth = authenticate_user("gold_customer", USER_PASSWORD)
if test_auth["success"]:
    claims = decode_token_claims(test_auth["id_token"])
    print(f"Auth successful!")
    print(f"  Customer ID: {claims.get('custom:customer_id')}")
    print(f"  Tier:        {claims.get('custom:tier')}")
    #print(f"  Token:       {test_auth['id_token'][:50]}...")    
else:
    print(f"Auth failed: {test_auth['error']}")

## Step 4: Harness Invocation Helper

Define `stream_orchestrator()` — calls `invoke_harness` with the user's `actorId` and yields text deltas for incremental UI rendering.

In [ ]:
import re

agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION)


def redact_sensitive(text: str) -> str:
    """Redact sensitive patterns (SSN, credit card, email) before streaming to the UI."""
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', 'XXX-XX-XXXX', text)
    text = re.sub(r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b', 'XXXX-XXXX-XXXX-XXXX', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[REDACTED-EMAIL]', text)
    return text


def stream_orchestrator(user_message: str, actor_id: str, session_id: str = None):
    """Stream the orchestrator harness response token-by-token.
    
    invoke_harness returns a stream of events. We yield each text fragment
    so the UI can render incrementally — no spinner that hides progress.
    The harness handles tool calls autonomously; this generator only yields
    final-answer text deltas.
    
    actor_id scopes long-term memory (preferences, facts) per user. In
    production this is the Cognito custom:customer_id claim; here we
    accept it as an explicit argument so the test cell below can demo
    cross-user isolation.
    """
    if not session_id:
        session_id = str(uuid.uuid4()).upper()

    messages = [{"role": "user", "content": [{"text": user_message}]}]

    response = agentcore_runtime.invoke_harness(
        harnessArn=HARNESS_ARN,
        runtimeSessionId=session_id,
        actorId=actor_id,
        messages=messages,
    )

    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            text = delta.get("text")
            if text:
                yield redact_sensitive(text)
        elif "runtimeClientError" in event:
            yield f"\n\n\u26a0\ufe0f Error: {event['runtimeClientError']['message']}"
            return

print(f"Orchestrator ready. Harness ARN: {HARNESS_ARN}")

# COMPLIANCE NOTE: This redaction applies only to UI output. If handling regulated data
# (HIPAA PHI, PCI card data, GDPR personal data), implement redaction at ingestion,
# apply encryption at rest, and ensure audit logging meets regulatory requirements.
# See AWS compliance resources: https://aws.amazon.com/compliance/


## Step 5: Launch the Streamlit Chatbot UI

Install Streamlit (if not already installed from Step 1).

In [ ]:
%pip install --quiet streamlit==1.54.0


Launch via the Jupyter proxy (SageMaker AI / Studio environment). Opens the chatbot at `/jupyterlab/default/proxy/8501/`.

In [ ]:
import subprocess
import os
from IPython.display import display, HTML

os.chdir(os.path.dirname(os.path.abspath('chatbot_app.py')) or '.')

# Display a clickable link that routes through the Jupyter proxy (same auth session)
display(HTML(
    f'<h3>👉 <a href="/jupyterlab/default/proxy/8501/" target="_blank">'
    f'Launch the Chatbot UI</a> and sign in with the credentials from SSM</h3>'
    f'<p>If the link does not work, wait a few seconds for Streamlit to start, then click again.</p>'
))

print(f"Username: gold_customer | silver_customer | bronze_customer")
#The password is stored as an SSM Parameter

print("Press Ctrl+C or restart the kernel to stop.\n")
# Launch Streamlit silently in the background
subprocess.run(["streamlit", "run", "chatbot_app.py", "--server.headless", "true"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


In [ ]:
!streamlit run chatbot_app.py --server.headless true

Launch directly (Kiro / local IDE environment). Uncomment and run — Streamlit will print the local URL.

In [ ]:
# # # Launch the Streamlit app
# # # This will open a new browser tab with the chatbot UI
# import subprocess
# import os

# os.chdir(os.path.dirname(os.path.abspath('chatbot_app.py')) or '.')
# print("Starting Streamlit app...")
# print("Open the URL shown below in your browser.")
# print(f"Username: gold_customer | silver_customer | bronze_customer")
# print(f"Password: {USER_PASSWORD}")
# print("Press Ctrl+C or restart the kernel to stop.\n")
# !streamlit run chatbot_app.py --server.headless true